In [1]:
import os
import random
import json
import asyncio
import nest_asyncio
from openai import AsyncOpenAI
from tqdm.notebook import tqdm
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
nest_asyncio.apply()

In [3]:
client = AsyncOpenAI()

In [4]:
semaphore = asyncio.Semaphore(5)

In [30]:
async def generate_questions_for_file(filepath: str, n_questions: int = 2) -> list[dict]:
    """
    Generate N quqesion-answer pairs for a given event file using gpt-4.1
    returns a list of dicts with question, answer and filename
    """
    async with semaphore:
        with open(filepath, 'r', encoding='utf-8') as f:
            content = f.read()

        filename = os.path.basename(filepath)

        prompt = f"""Given the following event file content, generate {n_questions} query-answer pairs.

                    Requirements:
                    - Queries should be one sentence search queries looking for events
                        - They should be framed as a general search intent (the query reader will not see the event file OR title OR any other information about the event)
                        - Queries should be what a user would type when searching for this type of event without knowing it exists
                        - Include event type, time period and the location
                    - Answers should contain the full event details: title, date, location
                    - All queries and answers must be in Polish language
                    - Return as a JSON object with a "questions" list containing dicts with "question" and "answer" fields

                    Event content:
                    {content}

                    Schema: 
                    {{
                        "questions": [
                            {{
                                "question": "search query text",
                                "answer": "Title: [event title]\\nDate: [date and time]\\nLocation: [location]"
                            }},
                            ...
                        ]
                    }}

                    Return ONLY the JSON object, no other text."""

        response = await client.chat.completions.create(
            model="gpt-4.1-mini",
            messages=[
                {"role": "system", "content": "You are a helpful assistant that generates factual question-answer pairs from event files"},
                {"role": "user", "content": prompt}
            ],
            response_format={"type": "json_object"},
            temperature=0.7)
        try:
            response_content = response.choices[0].message.content
            if not response_content:
                return []
            response_json = json.loads(response_content)
            if isinstance(response_json, list):
                qa_pairs = response_json
            elif isinstance(response_json, dict):
                qa_pairs = response_json.get("pairs", response_json.get("questions", response_json.get("data", []))) 
                if not isinstance(qa_pairs, list):
                    return []
            else:
                qa_pairs = []
        except json.JSONDecodeError as e:
            print(f"JSON decode error: {e}")
            return []
        except Exception as e:
            print(f"Unexpected error: {e}")
            return []

        results = []
        for pair in qa_pairs:
            results.append({
                "question": pair["question"],
                "answer": pair["answer"],
                "filename": filename
            })
        
        return results

async def generate_random_questions(n_pages: int = 5, questions_per_page: int = 2) -> list[dict]:
    """
    Generate random questions from a set of event files
    """
    event_files = [f for f in os.listdir("./data/events") if f.endswith(".md")]
    selected_files = random.sample(event_files, min(n_pages, len(event_files)))
    tasks = []
    for filename in selected_files:
        filepath = os.path.join("./data/events", filename)
        tasks.append(generate_questions_for_file(filepath, questions_per_page))
    all_results = []
    with tqdm(total=len(tasks), desc="Generating questions") as pbar:
        for coro in asyncio.as_completed(tasks):
            try:
                result = await coro
                all_results.append(result)
                pbar.update(1)
            except Exception as e:
                pbar.update(1)
                continue

    all_questions = []
    for questions in all_results:
        all_questions.extend(questions)
    return all_questions

async def main():
    n_pages = 50
    questions = await generate_random_questions(n_pages=n_pages, questions_per_page=1)
    print(f"\nGenerated {len(questions)} total questions")

    for i, q in enumerate(questions): 
        print(f"\n{i+1}. Q: {q['question']}")
        print(f"   A: {q['answer']}")
        print(f"   File: {q['filename']}")
    
    return questions

questions = await main()

Generating questions:   0%|          | 0/50 [00:00<?, ?it/s]


Generated 50 total questions

1. Q: Jakie są spektakle muzyczne dla dzieci w Warszawie latem 2025?
   A: Title: Muzyczne ZOO - spektakl dla dzieci / Teatr Katarynka
Date: 24.07.2025 18:00 - 20:00
Location: Teren rekreacyjny, Frontowa, Warszawa
   File: muzyczne_zoo_spektakl_dla_dzieci_teatr_katarynka_00409.md

2. Q: Jakie koncerty patriotyczne odbędą się we wrześniu 2025 roku w Warszawie?
   A: Title: Koncert patriotyczny
Date: 27.09.2025 19:30 - 27.09.2025 21:00
Location: Sanktuarium św. Andrzeja Boboli, ul. Rakowiecka 61, Warszawa
   File: koncert_patriotyczny_00162.md

3. Q: letnie kino plenerowe w Warszawie w 2025 roku w Ursusie
   A: Title: Letnie Kino Plenerowe w Ursusie
Date: 04.07.2025 21:30 - 04.07.2025 23:00 oraz kolejne piątki lipca i sierpnia 2025
Location: Park Hassów i Eko Park, Wojciechowskiego, 02-495 Warszawa
   File: letnie_kino_plenerowe_w_ursusie_00138.md

4. Q: Jakie są wydarzenia związane z indywidualnym zwiedzaniem wystaw historycznych w Warszawie w czerwcu 2025

In [ ]:
with open("generated_QUERIES_Polish_16_07.json", "w", encoding="utf-8") as f:
    json.dump(questions, f, indent=2, ensure_ascii=False)

In [32]:
questions

[{'question': 'Jakie są spektakle muzyczne dla dzieci w Warszawie latem 2025?',
  'answer': 'Title: Muzyczne ZOO - spektakl dla dzieci / Teatr Katarynka\nDate: 24.07.2025 18:00 - 20:00\nLocation: Teren rekreacyjny, Frontowa, Warszawa',
  'filename': 'muzyczne_zoo_spektakl_dla_dzieci_teatr_katarynka_00409.md'},
 {'question': 'Jakie koncerty patriotyczne odbędą się we wrześniu 2025 roku w Warszawie?',
  'answer': 'Title: Koncert patriotyczny\nDate: 27.09.2025 19:30 - 27.09.2025 21:00\nLocation: Sanktuarium św. Andrzeja Boboli, ul. Rakowiecka 61, Warszawa',
  'filename': 'koncert_patriotyczny_00162.md'},
 {'question': 'letnie kino plenerowe w Warszawie w 2025 roku w Ursusie',
  'answer': 'Title: Letnie Kino Plenerowe w Ursusie\nDate: 04.07.2025 21:30 - 04.07.2025 23:00 oraz kolejne piątki lipca i sierpnia 2025\nLocation: Park Hassów i Eko Park, Wojciechowskiego, 02-495 Warszawa',
  'filename': 'letnie_kino_plenerowe_w_ursusie_00138.md'},
 {'question': 'Jakie są wydarzenia związane z indyw